## MRE metadata crawler
Goes through the raw DICOMs of each scanner (one config per scanner: `configs/mre_7T.py`, `configs/mre_3T.py`) and collects the metadata of all MRE scans into one summary sheet per scanner.

How it works:
- A sequence folder counts as MRE if its DICOMs have an `ImageComments` tag starting with `MRE:` (MRE phase images). All other sequences are skipped.
- One row per MRE phase series (subject, date, series number) with: resolution, TR, TE, series description, slice plane, phase encoding direction, the MRE parameters from the image comment (frequencies, time steps, directions, motion encoding gradient vector) and the number of files.

Output:
- *MRE_metadata_summary_&lt;scanner&gt;.tsv* - in the output folder of each scanner (`out_dir`)

Run modes (arguments of `update_summary`):
- default - only subjects that are not in the sheet yet
- `subject_override=[...]` - only these subjects
- `force_all_subjects=True` - re-scan all subjects
- `dry_run=True` - compute everything, but don't write. Good to check first.
 
Manual columns are safe (e.g. Notes, PrivateName)
- Before every write, the previous sheet is copied to `backups/<name>_<date>_<time>.tsv`. Backups are never deleted automatically.
- NB! `overwrite=True` (or `append_if_exists=False`) replaces the whole sheet with only the subjects scanned in this run (a backup is still made).

In [ ]:
from seqinspect import load_config
from seqinspect.mre import crawler_from_config

In [ ]:
# label: used in the file name MRE_metadata_summary_<label>.tsv
crawlers = {
    "7T": crawler_from_config(load_config("configs/mre_7T.py"), label="7T"),
    "3T": crawler_from_config(load_config("configs/mre_3T.py"), label="3T"),
}
for crawler in crawlers.values():
    crawler.print_setup()
    print()

# Run modes:
# - default: only process subjects missing from the TSV
# - subject_override=[...] : process only those subjects
# - force_all_subjects=True : re-scan the complete DICOM tree

In [ ]:
# Set force_all_subjects=True to force a complete re-run for this scanner.
summary3T, detailed3T, updated_summary3T = crawlers["3T"].update_summary(
    verbose=True,
    dry_run=True,
    force_all_subjects=False,
)
updated_summary3T


In [ ]:
subject_override_7T = ["phantom_custom_fmre", "P7T_021_fmre_pilot1", "P7T_021_047_02", "P7T_021_048_02"]  # set to None to use missing-subject mode
subject_override_7T = None
summary7T, detailed7T, updated_summary7T = crawlers["7T"].update_summary(
    verbose=True,
    dry_run=False,
    subject_override=subject_override_7T,
    force_all_subjects=False,
)
updated_summary7T


In [ ]:
updated_summary7T[:-20]